# Applying demand tariffs to a btm site
This example shows how to build demand tariffs out of a combination of demand charges. It also shows the two ways of defining time periods over which the tariff applies.

## Build the model (see btm_battery_example for more details on this process)

In [185]:
import numpy as np
from echo.echo_models import *
from echo.echo_optimiser import *
from echo.objectives import *

In [186]:
time_periods = 96  # total number of intervals
interval_duration = 15  # Duration in mins of each interval
expansion_periods = 1  # Number of planning intervals - in echo V1, set to 1 always

In [187]:
load_array = np.array(
    [2.13, 2.09, 2.3, 2.11, 2.2, 2.23, 15, 15, 15, 2.19, 2.19, 2.19, 2.12, 2.15, 2.25, 2.12, 2.21, 2.16,
     2.26, 2.13, 2.08, 2.15, 2.42, 2.02, 2.3, 2.26, 2.35, 2.55, 3.23, 2.98, 3.49, 3.5, 3.12, 3.52, 3.94, 3.55,
     3.99, 3.71, 3.38, 3.76, 3.71, 3.78, 3.29, 3.65, 3.61, 3.75, 3.38, 3.66, 3.56, 3.69, 3.3, 3.61, 3.71, 3.82,
     3.17, 3.69, 3.74, 3.86, 3.57, 3.55, 3.75, 3.6, 3.67, 3.48, 3.51, 3.46, 3.19, 3.38, 3.19, 3.38, 3.04, 3.12,
     2.91, 3.11, 3.13, 2.77, 2.24, 2.54, 2.24, 2.24, 2.09, 2.33, 2.17, 2.16, 1.97, 2.16, 2.21, 2.18, 2.01, 2.16,
     2.19, 2.11, 2.17, 2.13, 12, 12])

pv_array = 2 * np.array(
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.05, 0.23, 0.52,
     0.74, 0.71, 0.63, 0.68, 0.97, 0.01, 0.52, 0.83, 0.83, 0.79, 1.22, 1.36, 1.27, 1.42, 1.97, 2.56, 2.91, 3.24,
     3.8, 4.3, 4.62, 4.84, 4.6, 4.17, 3.77, 3.76, 3.38, 2.64, 1.96, 1.76, 1.85, 2.4, 3.82, 5.13, 4.97, 5.02, 5.43,
     5.32, 3.56, 1.75, 1.43, 1.65, 1.69, 2.3, 2.71, 2.41, 2.63, 2.6, 1.9, 0.78, 0.13, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
     0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

pv_array *= -1  # use positive load convention, so solar gen is negative

In [188]:
system = OptimisationGraph()

In [ ]:
grid = Node() # create node representing upstream grid
grid.add_named_electrical_ports(['grid'])

In [ ]:
connection_point = ElectricalTellegenNode()     # create the connection point node
connection_point.add_named_electrical_ports(['load', 'inv', 'grid'])  # add ports with easily referenced names

In [ ]:
load = Node()                       # create a node to represent the load
load_port = ElectricalDemand()             # create an electrical demand port to attach to this node
load_port.add_demand_profile_from_array(load_array, expansion_periods)
load.ports['load'] = load_port # explicitly add the load_port to the dictionary of node ports

In [ ]:
inverter = Inverter(max_import=None,
                    max_export=None,
                    dc_ac_efficiency=1,
                    ac_dc_efficiency=1)

inverter.add_ac_port('inv')     # add a port with name 'inv' that is used to connect upstream/ac side
inverter.add_dc_port('battery')    # add a port with name 'battery' to connect to the battery
inverter.add_dc_port('pv')      # add a port with name 'pv' for us to connect the pv node into

In [ ]:
# create a node for the battery
battery = Node()
# create an electrical storage object
battery_port = ElectricalStorage(max_capacity=15.0,                # max capacity of battery in kwh
                       depth_of_discharge_limit=0,      # allowable depth of discharge in range [0,100] (i.e. percent)
                       charging_power_limit=1.25,       # max charging rate in kW
                       discharging_power_limit=-1.25,   # max discharging rate in kW
                       charging_efficiency=1,           # charging efficiency in range [0,1]
                       discharging_efficiency=1,        # discharging efficiency in range [0,1]
                       initial_state_of_charge=0.0)     # initial state of charge in kWh

battery.ports['battery'] = battery_port

In [ ]:
# create a node for the solar
solar = Node()
solar_port = ElectricalGeneration()     # create an electrical generation object
solar_port.curtailable = False          # set whether this can be curtailed or not
solar_port.add_generation_profile_from_array(pv_array, expansion_periods)
solar.ports['pv'] = solar_port        # add the solar port to the dictionary of node ports

In [ ]:
system.add_node_obj([grid, connection_point, load, inverter, solar, battery])  # nodes can be added one by one or as a list

In [ ]:
# Add edges to graph
system.connect_ports_and_create_edge(grid.ports['grid'], connection_point.ports['grid'])
system.connect_ports_and_create_edge(connection_point.ports['load'], load.ports['load'])
system.connect_ports_and_create_edge(connection_point.ports['inv'], inverter.ports['inv'])
system.connect_ports_and_create_edge(inverter.ports['battery'], battery.ports['battery'])
system.connect_ports_and_create_edge(inverter.ports['pv'], solar.ports['pv'])

## Create objectives
Here we will create an import demand tariff, which consists of two demand charges - a shoulder charge, and a peak period charge.

## Define peak period demand charge
A demand charge has two components - a rate (in $ per kW), and a window, which indicates the window over which the max demand is calculated.
If the charge applies over multiple windows the maximum over those multiple windows is calculated.
There is not currently functionality for a reset of the charge...

In [ ]:
# create an import demand tariff
# peak usage
peak_rate = 2.0  # the rate is applied per kW
peak_window = [0]*28 + [1]*8 + [0]*36 + [1]*12 + [0]*12  # binary array where 1 indicates when the charge applies

peak_charge = DemandCharge(rate=peak_rate,
                           window_array=peak_window,
                           min_demand=0.0)  # min demand can be used to move the 'floor' on the demand calculation.

## Define an off peak demand charge
Same process as above

In [ ]:
shoulder_rate = 1.0
shoulder_window = [0]*36 + [1]*36 + [0]*12 + [1]*8 + [0]*4

shoulder_charge = DemandCharge(rate=shoulder_rate,
                               window_array=shoulder_window,
                               min_demand=0.0)

### Create a demand tariff echo objective by combining these demand charges

In [ ]:
demand_tariff = ImportDemandTariffObjective(component=connection_point.ports['grid'],
                                            demand_charges=[peak_charge,
                                                            shoulder_charge])

# Create echo objective set which is a list of objectives
objective_set = ObjectiveSet(objective_list=[demand_tariff])


## Run the optimiser

In [ ]:
# Invoke the optimiser
optimiser = EchoOptimiser(interval_duration=interval_duration,
                          number_of_intervals=time_periods,
                          number_of_expansion_intervals=expansion_periods,
                          discount_rate=0,
                          ES=system,
                          objective_set=objective_set,
                          optimiser_engine='cplex')

# Optimise
optimiser.optimise()